# Calibration

## Load spatial references points

In [63]:
import pandas as pd
import os
import numpy as np
from Sequential_Fish.tools.utils import open_image

REF_CHANNEL = 0
CORR_CHANNEL = 1
DEGREE = 2
MAX_DISTANCE = 2000
VOXEL_SIZE = 200,97,97
VOXEL_SIZE = np.array([VOXEL_SIZE])


image_location = "/media/SSD_floricslimani/Fish_seq/Davide/Chromatic abberations/"
filename = "Beads-Field_01"
channels = [0,1,2]

In [64]:
df = pd.DataFrame()
for chan in channels :
    coordinates = pd.read_csv(image_location + f"{filename}_channel{chan}.csv", sep=";")
    coordinates.loc[:,['channel']] = chan
    df = pd.concat([
        df,
        coordinates,
    ], axis=0)
df = df.loc[:,['coordinates', 'channel']]
df

,coordinates,channel
0,"(23, 90, 1872)",0
1,"(23, 108, 1884)",0
2,"(23, 1918, 4)",0
3,"(24, 62, 1764)",0
4,"(24, 80, 1734)",0
...,...,...
513,"(43, 338, 1250)",2
514,"(43, 1146, 1605)",2
515,"(45, 1488, 942)",2
516,"(46, 923, 1042)",2


In [65]:
df['new_coordinates'] = df['coordinates'].str.replace("(","").str.replace(")","").str.split(',')
df['z_nm'] = df['new_coordinates'].apply(lambda x : int(x[0])*VOXEL_SIZE[0,0])
df['y_nm'] = df['new_coordinates'].apply(lambda x : int(x[2])*VOXEL_SIZE[0,1])
df['x_nm'] = df['new_coordinates'].apply(lambda x : int(x[1])*VOXEL_SIZE[0,2])

df['z'] = df['new_coordinates'].apply(lambda x : int(x[0]))
df['y'] = df['new_coordinates'].apply(lambda x : int(x[2]))
df['x'] = df['new_coordinates'].apply(lambda x : int(x[1]))


df['coordinates_nm'] = list(zip(df['z_nm'], df['y_nm'], df['x_nm'],))
df['coordinates'] = list(zip(df['z'], df['y'], df['x'],))
df = df.drop("new_coordinates", axis=1)
df

,coordinates,channel,z_nm,y_nm,x_nm,z,y,x,coordinates_nm
0,"(23, 1872, 90)",0,4600,181584,8730,23,1872,90,"(4600, 181584, 8730)"
1,"(23, 1884, 108)",0,4600,182748,10476,23,1884,108,"(4600, 182748, 10476)"
2,"(23, 4, 1918)",0,4600,388,186046,23,4,1918,"(4600, 388, 186046)"
3,"(24, 1764, 62)",0,4800,171108,6014,24,1764,62,"(4800, 171108, 6014)"
4,"(24, 1734, 80)",0,4800,168198,7760,24,1734,80,"(4800, 168198, 7760)"
...,...,...,...,...,...,...,...,...,...
513,"(43, 1250, 338)",2,8600,121250,32786,43,1250,338,"(8600, 121250, 32786)"
514,"(43, 1605, 1146)",2,8600,155685,111162,43,1605,1146,"(8600, 155685, 111162)"
515,"(45, 942, 1488)",2,9000,91374,144336,45,942,1488,"(9000, 91374, 144336)"
516,"(46, 1042, 923)",2,9200,101074,89531,46,1042,923,"(9200, 101074, 89531)"


In [66]:
coordinates = {chan : df.loc[df['channel'] == chan,['z','y','x']].to_numpy().astype(int) for chan in channels }
coordinates_nm = {chan : df.loc[df['channel'] == chan,['z_nm','y_nm','x_nm']].to_numpy().astype(int) for chan in channels }

## Match beads

In [67]:
from sklearn.neighbors import NearestNeighbors
def match_beads(coords1, coords2, max_dist=5):
    """Match nearest beads between two channels."""
    nn = NearestNeighbors(n_neighbors=1).fit(coords2)
    distances, indices = nn.kneighbors(coords1)
    matches = distances[:, 0] < max_dist
    return coords1[matches], coords2[indices[matches, 0]]

In [68]:
coords, dist = match_beads(
    coordinates_nm[CORR_CHANNEL],
    coordinates_nm[REF_CHANNEL],
    max_dist=MAX_DISTANCE
    )

print("total spot : ", len(coordinates[CORR_CHANNEL]))
print("total spot_ref : ", len(coordinates[REF_CHANNEL]))
print("match found : ", len(coords))

total spot :  532
total spot_ref :  519
match found :  525


## Fit polynomial

In [69]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from scipy.ndimage import map_coordinates

def fit_polynomial_transform_3d(src_points, dst_points, degree=2):
    """Fit 3D polynomial regression mapping coords → dst."""
    poly = PolynomialFeatures(degree)
    X_poly = poly.fit_transform(src_points)
    model_x = LinearRegression().fit(X_poly, dst_points[:, 2])  # x
    model_y = LinearRegression().fit(X_poly, dst_points[:, 1])  # y
    model_z = LinearRegression().fit(X_poly, dst_points[:, 0])  # z
    return poly, model_x, model_y, model_z

In [70]:
poly, model_x, model_y, model_z = fit_polynomial_transform_3d(
    coords,
    dist, 
    degree=DEGREE
)

In [71]:
inv_poly, inv_model_x, inv_model_y, inv_model_z = fit_polynomial_transform_3d(
    dist, 
    coords,
    degree=DEGREE
)

## Correct abberations

In [72]:
def apply_polynomial_transform_3d(volume, poly, model_x, model_y, model_z):
    """Warp 3D volume using learned polynomial transform."""
    z, y, x = volume.shape
    zz, yy, xx = np.meshgrid(np.arange(z), np.arange(y), np.arange(x), indexing='ij')
    coords = np.stack([zz.ravel(), yy.ravel(), xx.ravel()], axis=1)
    print(coords.shape)

    X_poly = poly.transform(coords * VOXEL_SIZE)
    new_z_nm = model_z.predict(X_poly)
    new_y_nm = model_y.predict(X_poly)
    new_x_nm = model_x.predict(X_poly)

    #convert back to pixel
    new_coords_pixel = np.stack([new_z_nm, new_y_nm, new_x_nm], axis=0) / VOXEL_SIZE.T

    warped = map_coordinates(volume, new_coords_pixel, order=1, mode='reflect').reshape(z, y, x)
    return warped

In [73]:
image = open_image(image_location + "/Beads-Field_01.tif")
image.shape

(47, 1981, 2004, 3)

In [74]:
image_corrected = apply_polynomial_transform_3d(
    image[...,CORR_CHANNEL],
    poly=poly,
    model_x=inv_model_x,
    model_y=inv_model_y,
    model_z=inv_model_z
)
image_corrected.shape

(186586428, 3)


(47, 1981, 2004)

## Napari

In [92]:
import napari
os.environ["QT_QPA_PLATFORM"] = "xcb"
Viewer = napari.Viewer()
Viewer.add_image(image[...,REF_CHANNEL], name= "ref", scale=(200,97,97), blending='additive', colormap = 'green', interpolation2d= 'cubic')
Viewer.add_image(image[...,CORR_CHANNEL], name= "uncorrected", scale=(200,97,97), blending='additive', colormap = 'red', interpolation2d= 'cubic')
Viewer.add_image(image_corrected, name= "corrected", scale=(200,97,97), blending='additive', colormap = 'blue', visible=False, interpolation2d= 'cubic')

pass

# Validation

In [76]:
import pandas as pd
import os
import numpy as np
from Sequential_Fish.tools.utils import open_image

REF_CHANNEL = 0
CORR_CHANNEL = 1
DEGREE = 2
MAX_DISTANCE = 2000
VOXEL_SIZE = 200,97,97
VOXEL_SIZE = np.array([VOXEL_SIZE])


image_location = "/media/SSD_floricslimani/Fish_seq/Davide/Chromatic abberations/"
filename = "Beads-Field_02"
channels = [0,1,2]

In [77]:
image_2 = open_image(image_location + f"/{filename}.tif")
image_2.shape

(36, 1981, 2004, 3)

In [78]:
df = pd.DataFrame()
for chan in channels :
    coordinates = pd.read_csv(image_location + f"{filename}_channel{chan}.csv", sep=";")
    coordinates.loc[:,['channel']] = chan
    df = pd.concat([
        df,
        coordinates,
    ], axis=0)
df = df.loc[:,['coordinates', 'channel']]
df

,coordinates,channel
0,"(10, 1847, 1832)",0
1,"(11, 52, 1726)",0
2,"(11, 78, 1884)",0
3,"(11, 164, 27)",0
4,"(11, 292, 221)",0
...,...,...
431,"(26, 1129, 814)",2
432,"(26, 1184, 635)",2
433,"(27, 989, 657)",2
434,"(27, 1129, 1084)",2


In [79]:
df['new_coordinates'] = df['coordinates'].str.replace("(","").str.replace(")","").str.split(',')
df['z_nm'] = df['new_coordinates'].apply(lambda x : int(x[0])*VOXEL_SIZE[0,0])
df['y_nm'] = df['new_coordinates'].apply(lambda x : int(x[2])*VOXEL_SIZE[0,1])
df['x_nm'] = df['new_coordinates'].apply(lambda x : int(x[1])*VOXEL_SIZE[0,2])

df['z'] = df['new_coordinates'].apply(lambda x : int(x[0]))
df['y'] = df['new_coordinates'].apply(lambda x : int(x[2]))
df['x'] = df['new_coordinates'].apply(lambda x : int(x[1]))


df['coordinates_nm'] = list(zip(df['z_nm'], df['y_nm'], df['x_nm'],))
df['coordinates'] = list(zip(df['z'], df['y'], df['x'],))
df = df.drop("new_coordinates", axis=1)
df

,coordinates,channel,z_nm,y_nm,x_nm,z,y,x,coordinates_nm
0,"(10, 1832, 1847)",0,2000,177704,179159,10,1832,1847,"(2000, 177704, 179159)"
1,"(11, 1726, 52)",0,2200,167422,5044,11,1726,52,"(2200, 167422, 5044)"
2,"(11, 1884, 78)",0,2200,182748,7566,11,1884,78,"(2200, 182748, 7566)"
3,"(11, 27, 164)",0,2200,2619,15908,11,27,164,"(2200, 2619, 15908)"
4,"(11, 221, 292)",0,2200,21437,28324,11,221,292,"(2200, 21437, 28324)"
...,...,...,...,...,...,...,...,...,...
431,"(26, 814, 1129)",2,5200,78958,109513,26,814,1129,"(5200, 78958, 109513)"
432,"(26, 635, 1184)",2,5200,61595,114848,26,635,1184,"(5200, 61595, 114848)"
433,"(27, 657, 989)",2,5400,63729,95933,27,657,989,"(5400, 63729, 95933)"
434,"(27, 1084, 1129)",2,5400,105148,109513,27,1084,1129,"(5400, 105148, 109513)"


In [80]:
coordinates = {chan : df.loc[df['channel'] == chan,['z','y','x']].to_numpy().astype(int) for chan in channels }
coordinates_nm = {chan : df.loc[df['channel'] == chan,['z_nm','y_nm','x_nm']].to_numpy().astype(int) for chan in channels }

## Applying correction to signal

In [90]:
signal_corrected = apply_polynomial_transform_3d(
        image_2[...,CORR_CHANNEL],
        poly=poly,
        model_x = inv_model_x,
        model_y = inv_model_y,
        model_z = inv_model_z,        
    )

(142917264, 3)


### napari

In [91]:
import napari
os.environ["QT_QPA_PLATFORM"] = "xcb"
Viewer = napari.Viewer()


Viewer.add_image(image_2[...,REF_CHANNEL], name= f"raw signal {chan}", scale=(200,97,97), blending='additive', colormap = 'green', interpolation2d= 'cubic')
Viewer.add_image(image_2[...,CORR_CHANNEL], name= f"uncorrected signal {chan}", scale=(200,97,97), blending='additive', colormap = 'blue', interpolation2d= 'cubic')
Viewer.add_image(signal_corrected, name= f"corrected signal {chan}", scale=(200,97,97), blending='additive', colormap = 'red', interpolation2d= 'cubic')

pass

## Applying correction to detected spots

In [83]:
spots_ref = coordinates[REF_CHANNEL]
spots_abberration = coordinates[CORR_CHANNEL]

In [84]:
def apply_polynomial_transform_3d_spots(coords, poly, model_x, model_y, model_z) : 

    monosomes = poly.transform(spots_abberration * VOXEL_SIZE)
    new_z_nm = model_z.predict(monosomes)
    new_y_nm = model_y.predict(monosomes)
    new_x_nm = model_x.predict(monosomes)
    new_coords_pixel = np.stack([new_z_nm, new_y_nm, new_x_nm], axis=1) / VOXEL_SIZE

    return new_coords_pixel


In [85]:
corrected_coordinates = apply_polynomial_transform_3d_spots(
    spots_abberration,
    poly=poly,
    model_x=model_x,
    model_y=model_y,
    model_z=model_z,
)
corrected_coordinates.shape

(443, 3)

### napari

In [93]:
import napari
os.environ["QT_QPA_PLATFORM"] = "xcb"
Viewer = napari.Viewer()


Viewer.add_image(image_2[...,REF_CHANNEL], name= f"raw signal", scale=(200,97,97), blending='additive', colormap = 'green', interpolation2d= 'cubic')
Viewer.add_image(image_2[...,CORR_CHANNEL], name= f"uncorrected signal ", scale=(200,97,97), blending='additive', colormap = 'blue', interpolation2d= 'cubic')
Viewer.add_image(signal_corrected, name= f"corrected signal", scale=(200,97,97), blending='additive', colormap = 'red', interpolation2d= 'cubic')

Viewer.add_points(spots_ref, name= "beads reference", scale=(200,97,97), blending='additive', face_color='transparent', symbol= 'disc', size=8)
Viewer.add_points(spots_abberration, name= "beads abberation", scale=(200,97,97), blending='additive', face_color='transparent', symbol= 'disc', size=8,border_color= 'red')
Viewer.add_points(corrected_coordinates, name= "beads corrected", scale=(200,97,97), blending='additive', face_color='transparent', symbol= 'cross', size=8)

pass